In [29]:
import numpy as np
import cv2
import pandas as pd
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [30]:
df = pd.read_csv("train.csv")

df_pivot = df.pivot(
    index="image_path",
    columns="target_name",
    values="target"
).reset_index()

df_pivot.head()

target_name,image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
0,train/ID1011485656.jpg,0.0000,31.9984,16.2751,48.2735,16.2750
1,train/ID1012260530.jpg,0.0000,0.0000,7.6000,7.6000,7.6000
2,train/ID1025234388.jpg,6.0500,0.0000,0.0000,6.0500,6.0500
3,train/ID1028611175.jpg,0.0000,30.9703,24.2376,55.2079,24.2376
4,train/ID1035947949.jpg,0.4343,23.2239,10.5261,34.1844,10.9605


In [31]:
X = []
y = []

for i in range(len(df_pivot)):

    img_path = df_pivot["image_path"][i]

    image = cv2.imread(img_path)
    image = cv2.resize(image,(224,224))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    image = preprocess_input(image)

    X.append(image)

    y.append([
        df_pivot["Dry_Clover_g"][i],
        df_pivot["Dry_Dead_g"][i],
        df_pivot["Dry_Green_g"][i],
        df_pivot["Dry_Total_g"][i],
        df_pivot["GDM_g"][i]
    ])

X = np.array(X)
y = np.array(y)

In [32]:
X.shape

(357, 224, 224, 3)

In [33]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [34]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

In [35]:
train_generator = datagen.flow(
    X_train,
    y_train,
    batch_size=32
)

In [36]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)
for layer in base_model.layers:
    layer.trainable = False

In [37]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

In [38]:
x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)

predictions = Dense(5, activation="linear")(x)
model = Model(inputs=base_model.input, outputs=predictions)

In [39]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="mse",
    metrics=["mae"]
)

In [42]:
history = model.fit(
    train_generator,
    validation_data=(X_val, y_val),
    epochs=160
)

Epoch 1/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 365ms/step - loss: 355.5620 - mae: 12.5406 - val_loss: 285.5662 - val_mae: 12.1509
Epoch 2/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 356ms/step - loss: 360.2149 - mae: 12.5612 - val_loss: 284.3120 - val_mae: 12.1427
Epoch 3/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 358ms/step - loss: 365.0093 - mae: 12.4760 - val_loss: 282.8864 - val_mae: 12.1154
Epoch 4/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 369ms/step - loss: 354.7280 - mae: 12.3590 - val_loss: 282.0624 - val_mae: 12.1051
Epoch 5/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 358ms/step - loss: 341.1700 - mae: 12.3845 - val_loss: 280.8702 - val_mae: 12.0730
Epoch 6/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 363ms/step - loss: 345.8894 - mae: 12.4155 - val_loss: 279.9280 - val_mae: 12.0594
Epoch 7/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 368ms/step - loss: 336.8122 - mae: 12.3147 - val_loss: 279.5231 - val_mae: 12.0682
Epoch 8/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 359ms/step - loss: 342.4580 - mae: 12.2540 - val_loss: 278.1193 - val_mae: 12.0419
Epoch 9/100
9/9 

In [43]:
img_path = "test/ID1001187975.jpg"

image = cv2.imread(img_path)
image = cv2.resize(image,(224,224))
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

image = preprocess_input(image)

image = np.expand_dims(image, axis=0)

pred = model.predict(image)

print(pred)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step   
[[-1.1635356 21.752304  46.240673  67.62024   45.228832 ]]


In [44]:
submission = pd.read_csv("sample_submission.csv")

In [46]:
submission["biomass"] = submission["sample_id"].apply(lambda x: {
    "Dry_Clover_g": pred[0][0],
    "Dry_Dead_g": pred[0][1],
    "Dry_Green_g": pred[0][2],
    "Dry_Total_g": pred[0][3],
    "GDM_g": pred[0][4],
}[x.split("_",1)[1].strip("_")])

In [47]:
submission.to_csv("sub.csv", index=False)